In [4]:
# STEP 2 - CONNECT TO THE DATA  (run every time you open the notebook)
%pip install -U "zarr>=3"
import boto3, s3fs
import xarray as xr

DATA_FOLDER = "elnino"       # your team's data folder - no need to change this

PROFILE = "hack-team"            # use the key you saved in Step 1 (the hub's own AWS login can't see this data)
BUCKET  = "ocean-hackathon-2026"
fs = s3fs.S3FileSystem(profile=PROFILE)
opts = {"profile": PROFILE}

# Ask AWS which team this key belongs to, so you never have to type your team name
who = boto3.Session(profile_name=PROFILE).client("sts").get_caller_identity()["Arn"]
TEAM = who.split("/")[-1]                      # e.g. "team-sound"
TEAM_DIR = f"{BUCKET}/teams/{TEAM}"            # your team's workspace (you can save files here)
DATA_DIR = f"{BUCKET}/data/{DATA_FOLDER}"      # your raw data (read-only)
OUT_DIR  = f"{TEAM_DIR}/processed"

print("Connected as:", TEAM)
print("Raw data:    ", "s3://" + DATA_DIR)
print("Your outputs:", "s3://" + TEAM_DIR)
print(fs.ls(OUT_DIR))

import zarr
print(zarr.__version__)

ocean     = xr.open_zarr(f"s3://{OUT_DIR}/ocean_monthly.zarr",     storage_options=opts)
clim      = xr.open_zarr(f"s3://{OUT_DIR}/climatology.zarr",       storage_options=opts)
wind      = xr.open_zarr(f"s3://{OUT_DIR}/wind_monthly.zarr",      storage_options=opts)
wind_clim = xr.open_zarr(f"s3://{OUT_DIR}/wind_climatology.zarr",  storage_options=opts)

stores = {"ocean": ocean, "clim": clim, "wind": wind, "wind_clim": wind_clim}

for name, ds in stores.items():
    print(f"\n=== {name} ===  dims: {dict(ds.sizes)}")
    for v, da in ds.data_vars.items():
        units = da.attrs.get("units", "")
        print(f"  {v:24s} {str(da.dims):35s} {units}")

Note: you may need to restart the kernel to use updated packages.
Connected as: team-elnino
Raw data:     s3://ocean-hackathon-2026/data/elnino
Your outputs: s3://ocean-hackathon-2026/teams/team-elnino
['ocean-hackathon-2026/teams/team-elnino/processed/climatology.zarr', 'ocean-hackathon-2026/teams/team-elnino/processed/enso.json', 'ocean-hackathon-2026/teams/team-elnino/processed/meta.json', 'ocean-hackathon-2026/teams/team-elnino/processed/ocean_monthly.zarr', 'ocean-hackathon-2026/teams/team-elnino/processed/wind_climatology.zarr', 'ocean-hackathon-2026/teams/team-elnino/processed/wind_monthly.zarr']
3.4.1

=== ocean ===  dims: {'time': 404, 'latitude': 176, 'longitude': 240}
  mlotst                   ('time', 'latitude', 'longitude')   m
  so                       ('time', 'latitude', 'longitude')   0.001
  to                       ('time', 'latitude', 'longitude')   degrees_C
  ugo                      ('time', 'latitude', 'longitude')   m s-1
  vgo                      ('time', 